# Controlling an interferometer using EPICS and a stage using ophyd
awojdyla@lbl.gov, Feb 2026

In [1]:
# !pip install pyepics
# !pip install ophyd

In [2]:
#using base 3.13.5 on GW
import epics
import ophyd

## Reading the PV for the interferometer
We assume that an IOC is running for the Smaract interferomter, and that we can poll it through the network

In [3]:
epics.caget('CATERETE:PICOSCALE:POS_0')

-171677079

## Moving the stage

### using the python package

In [4]:
#!pip install newportxps

### create an ophyd wrapper

In [ ]:
from bluesky import RunEngine
import numpy as np
import bluesky.plans as bp
import bluesky.plan_stubs as bps
import time
from ophyd import Device, Component as Cpt, Signal
from ophyd import EpicsSignal
from ophyd.status import SubscriptionStatus
import threading

In [ ]:
# Create a Run Engine
RE = RunEngine()

class NewportXPSMotor(Device):
    # Ophyd signals
    user_readback = Cpt(Signal, value=0, kind='hinted')
    user_setpoint = Cpt(Signal, value=0, kind='normal')
    
    def __init__(self, ip_address, stage_name, username='Administrator', 
                 password='Administrator', *args, **kwargs):
        super().__init__(*args, **kwargs)
        
        from newportxps import NewportXPS
        
        self.stage_name = stage_name
        self.group_name = stage_name.split('.')[0]  # Extract 'Group1' from 'Group1.Pos'
        self.xps = NewportXPS(ip_address, username=username, password=password)
        
        # Update initial position
        self._update_position()
        
    def _update_position(self):
        """Read current position from hardware"""
        pos = self.xps.read_stage_position(self.stage_name)
        self.user_readback.put(pos)
        return pos
    
    def _is_moving(self):
        """Check if the motor is currently moving"""
        status = self.xps.get_group_status()
        group_status = status.get(self.group_name, '')
        return 'Moving' in group_status
    
    def set(self, position, timeout=30):
        """Move to a position and return a status object"""
        
        def check_done(*, old_value, value, **kwargs):
            """Callback to check if motion is complete"""
            return not self._is_moving()
        
        # Start the move
        self.user_setpoint.put(position)
        self.xps.move_stage(self.stage_name, position)
        
        # Create a status object that monitors completion
        status = SubscriptionStatus(self.user_readback, check_done, timeout=timeout)
        
        # Start a background thread to update position during motion
        def monitor_motion():
            while not status.done:
                self._update_position()
                time.sleep(0.05)  # 50ms polling
            # Final position update
            self._update_position()
        
        thread = threading.Thread(target=monitor_motion, daemon=True)
        thread.start()
        
        return status
    
    def read(self):
        """Read current position"""
        pos = self._update_position()
        return {f'{self.name}_user_readback': {'value': pos, 'timestamp': time.time()}}
    
    def describe(self):
        """Describe the readback signal"""
        return {f'{self.name}_user_readback': {
            'source': f'XPS:{self.stage_name}',
            'dtype': 'number',
            'shape': []
        }}

In [7]:
#stock ophyd opbject for EPICS signal
interf = EpicsSignal('CATERETE:PICOSCALE:POS_0', name='interferometer_position')

#custom ophyd object
motor = NewportXPSMotor(
    ip_address='192.168.10.20',
    stage_name='Group1.Pos',
    username='Administrator',
    password='Administrator',
    name='xps_motor'
)

pos_inter_m = interf.get()*1e-12
pos_readback_m = motor.get()[0]*1e-3
print(f"Interferometer position: {pos_inter_m*1e3} mm")
print(f"Motor readback position: {pos_readback_m*1e3} mm")


Interferometer position: -0.171677079 mm
Motor readback position: 9.999994 mm


In [8]:
# from tiled.client import simple
# c = simple('../data/test')  # or just simple() for temporary/dev storage
# #ifndef 
# # c.create_container('raw', specs=['CatalogOfBlueskyRuns'])

# from bluesky_tiled_plugins import TiledWriter
# tw = TiledWriter(c['raw'])

# from bluesky import RunEngine
# RE = RunEngine()
# RE.subscribe(tw)


# RE(bp.scan([interf, motor], motor, -10, 10, 11))

## regular loop scan with ophyd objects

In [14]:
step_size_mm = 0.1
scan_length_mm = 10.0
positions_mm = np.arange(0, scan_length_mm + step_size_mm, step_size_mm)

In [15]:
positions_mm

array([ 0. ,  0.1,  0.2,  0.3,  0.4,  0.5,  0.6,  0.7,  0.8,  0.9,  1. ,
        1.1,  1.2,  1.3,  1.4,  1.5,  1.6,  1.7,  1.8,  1.9,  2. ,  2.1,
        2.2,  2.3,  2.4,  2.5,  2.6,  2.7,  2.8,  2.9,  3. ,  3.1,  3.2,
        3.3,  3.4,  3.5,  3.6,  3.7,  3.8,  3.9,  4. ,  4.1,  4.2,  4.3,
        4.4,  4.5,  4.6,  4.7,  4.8,  4.9,  5. ,  5.1,  5.2,  5.3,  5.4,
        5.5,  5.6,  5.7,  5.8,  5.9,  6. ,  6.1,  6.2,  6.3,  6.4,  6.5,
        6.6,  6.7,  6.8,  6.9,  7. ,  7.1,  7.2,  7.3,  7.4,  7.5,  7.6,
        7.7,  7.8,  7.9,  8. ,  8.1,  8.2,  8.3,  8.4,  8.5,  8.6,  8.7,
        8.8,  8.9,  9. ,  9.1,  9.2,  9.3,  9.4,  9.5,  9.6,  9.7,  9.8,
        9.9, 10. ])

In [19]:
for pos_mm in positions_mm:
    status = motor.set(pos_mm)
    status.wait()  # Wait for the move to complete
    pos_inter_m = interf.get()*1e-12
    pos_readback_m = motor.get()[0]*1e-3
    print(f"Set motor to {pos_mm} mm, interferometer reads {pos_inter_m*1e3} mm, motor readback is {pos_readback_m*1e3} mm")
# pos_xps_mm = 0

# motor.set(pos_xps_mm)  # Convert from pm to mm for motor setpoint
# sleep(1)  # Allow some time for the motor to move and settle

# pos_ps_au = interf.get()
# print(f"Set motor to {pos_xps_mm} mm, interferometer reads {pos_ps_au*1e-8} au")

Set motor to 0.0 mm, interferometer reads -0.171677079 mm, motor readback is -0.000381 mm
Set motor to 0.1 mm, interferometer reads -0.171677079 mm, motor readback is 0.09998400000000002 mm
Set motor to 0.2 mm, interferometer reads -0.171677079 mm, motor readback is 0.200013 mm
Set motor to 0.30000000000000004 mm, interferometer reads -0.171677079 mm, motor readback is 0.300029 mm
Set motor to 0.4 mm, interferometer reads -0.171677079 mm, motor readback is 0.400043 mm
Set motor to 0.5 mm, interferometer reads -0.171677079 mm, motor readback is 0.500047 mm
Set motor to 0.6000000000000001 mm, interferometer reads -0.171677079 mm, motor readback is 0.600053 mm
Set motor to 0.7000000000000001 mm, interferometer reads -0.171677079 mm, motor readback is 0.700047 mm
Set motor to 0.8 mm, interferometer reads -0.171677079 mm, motor readback is 0.800031 mm
Set motor to 0.9 mm, interferometer reads -0.171677079 mm, motor readback is 0.900031 mm
Set motor to 1.0 mm, interferometer reads -0.1716770